In [1]:
import numpy as np
import pandas as pd
from sktime.datasets import load_forecastingdata
from statsmodels.tsa.stattools import acf
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoTheta

c:\Users\lenusiker\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df, meta = load_forecastingdata('electricity_hourly_dataset')

Библиотека sktime позволяет загрузить датасет с Monash Time Series Forecasting Repository.

In [3]:
df

,series_name,start_timestamp,series_value
0,T1,2012-01-01 00:00:01,"[14.0, 18.0, 21.0, 20.0, 22.0, 20.0, 20.0, 20...."
1,T2,2012-01-01 00:00:01,"[69.0, 92.0, 96.0, 92.0, 91.0, 92.0, 91.0, 92...."
2,T3,2012-01-01 00:00:01,"[234.0, 312.0, 312.0, 312.0, 312.0, 187.0, 138..."
3,T4,2012-01-01 00:00:01,"[415.0, 556.0, 560.0, 443.0, 346.0, 340.0, 376..."
4,T5,2012-01-01 00:00:01,"[215.0, 292.0, 272.0, 213.0, 190.0, 178.0, 199..."
...,...,...,...
316,T317,2012-01-01 00:00:01,"[48.0, 65.0, 64.0, 65.0, 75.0, 64.0, 65.0, 69...."
317,T318,2012-01-01 00:00:01,"[38.0, 47.0, 43.0, 39.0, 40.0, 39.0, 49.0, 55...."
318,T319,2012-01-01 00:00:01,"[1558.0, 2177.0, 2193.0, 1315.0, 1378.0, 1250...."
319,T320,2012-01-01 00:00:01,"[182.0, 253.0, 218.0, 195.0, 191.0, 185.0, 191..."


Проверка сезонности

In [4]:
def get_series_features(values):
    
    values = np.asarray(values, dtype=float)
    acf_vals = acf(values, nlags=168, fft=True)

    return pd.Series({
        'length': len(values),
        'mean': np.mean(values),
        'std': np.std(values),
        'acf_24': acf_vals[24] if len(acf_vals) > 24 else np.nan,
        'acf_168': acf_vals[168] if len(acf_vals) > 168 else np.nan,
    })

In [5]:
series_features = pd.concat([df[['series_name']], df['series_value'].apply(get_series_features)], axis=1)

#смотрим на самые сезонные ряды
series_features.sort_values('acf_168', ascending=False).head(15)

,series_name,length,mean,std,acf_24,acf_168
292,T293,26304.0,4156.689895,2098.660315,0.980675,0.973433
306,T307,26304.0,2642.776916,1551.084124,0.975384,0.971380
314,T315,26304.0,10065.918491,6568.573136,0.983105,0.970790
88,T89,26304.0,2727.282923,1714.021024,0.979323,0.967235
182,T183,26304.0,721.947194,1338.440911,0.993150,0.966766
109,T110,26304.0,535.426589,290.891746,0.876906,0.966722
93,T94,26304.0,4498.884884,2551.594029,0.983968,0.964701
146,T147,26304.0,255.865610,186.484639,0.982326,0.962390
312,T313,26304.0,1165.538853,736.021677,0.971569,0.962244
175,T176,26304.0,8502.146099,2916.252395,0.975323,0.961496


In [6]:
#самые слабые
series_features.sort_values('acf_168', ascending=True).head(15)

,series_name,length,mean,std,acf_24,acf_168
113,T114,26304.0,260.534139,245.067607,0.535034,0.184915
121,T122,26304.0,95.987226,8.541787,0.360300,0.415137
118,T119,26304.0,335.851848,42.038673,0.856778,0.469764
84,T85,26304.0,105.589112,112.295901,0.491033,0.480766
2,T3,26304.0,16.821624,49.189442,0.626099,0.533119
0,T1,26304.0,23.263762,24.126706,0.751751,0.576605
298,T299,26304.0,10.627395,15.768540,0.737281,0.635553
106,T107,26304.0,210.717191,274.499281,0.467668,0.639931
9,T10,26304.0,227.069267,102.180629,0.564213,0.649295
83,T84,26304.0,106.849985,237.484910,0.598925,0.658088


Так как гипотеза предполагает, что может быть влияние, зависящее от силы сезонности, возьмем выборку из 60 рядов, по 20 на каждую степень (сильное, среднее, слабое влияние)

In [7]:
series_features['seasonality_group'] = pd.qcut(series_features['acf_168'], q=3, labels=['weak', 'medium', 'strong'])

In [8]:
series_features

,series_name,length,mean,std,acf_24,acf_168,seasonality_group
0,T1,26304.0,23.263762,24.126706,0.751751,0.576605,weak
1,T2,26304.0,112.885569,25.552655,0.775762,0.737003,weak
2,T3,26304.0,16.821624,49.189442,0.626099,0.533119,weak
3,T4,26304.0,440.335196,152.598150,0.894935,0.909569,medium
4,T5,26304.0,200.536724,69.726023,0.876298,0.809036,weak
...,...,...,...,...,...,...,...
316,T317,26304.0,350.702137,241.236858,0.952388,0.915400,medium
317,T318,26304.0,51.490344,37.228129,0.124143,0.763216,weak
318,T319,26304.0,2264.188641,545.269007,0.756778,0.855248,weak
319,T320,26304.0,507.008858,267.440389,0.492337,0.907023,medium


In [9]:
selected_series = (series_features.groupby('seasonality_group', group_keys=False) #группируем по силе сезонности
                   .apply(lambda x: x.sample(20, random_state=42)) #выбираем случайные 20 рядов из каждой группы
                   .sort_values(['seasonality_group', 'acf_168']) #сортируем по силе сезонности
                   .reset_index(drop=True)
)

selected_series.head(10)

C:\Users\lenusiker\AppData\Local\Temp\ipykernel_24988\3120288297.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  selected_series = (series_features.groupby('seasonality_group', group_keys=False) #группируем по силе сезонности
C:\Users\lenusiker\AppData\Local\Temp\ipykernel_24988\3120288297.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(20, random_state=42)) #выбираем случайные 20 рядов из каждой группы


,series_name,length,mean,std,acf_24,acf_168,seasonality_group
0,T1,26304.0,23.263762,24.126706,0.751751,0.576605,weak
1,T299,26304.0,10.627395,15.768540,0.737281,0.635553,weak
2,T7,26304.0,25.943659,27.391823,0.898070,0.670192,weak
3,T126,26304.0,135.455254,21.204412,0.841894,0.671536,weak
4,T71,26304.0,131.809421,60.837370,0.871179,0.693382,weak
5,T108,26304.0,60.077821,65.223491,0.469422,0.753090,weak
6,T83,26304.0,555.620096,148.422498,0.830074,0.800046,weak
7,T123,26304.0,111.984261,17.057128,0.905405,0.804513,weak
8,T128,26304.0,311.364545,127.772198,0.662935,0.833312,weak
9,T69,26304.0,143.546343,60.906677,0.683268,0.834702,weak


In [10]:
selected_ids = selected_series['series_name'].tolist()

df_raw = df.copy()
df_selected = df[df['series_name'].isin(selected_ids)]

In [27]:
TRAIN_LENGTH = 12 * 7 * 24   # 2016
HORIZON = 168
TOTAL_LENGTH = TRAIN_LENGTH + HORIZON

df_selected_cut = df_selected.copy()
df_selected_cut['series_value'] = df_selected_cut['series_value'].apply(lambda x: x[-TOTAL_LENGTH:])

В датасете каждая строка соответствует одному временному ряду, значения хранятся в списке в колонке series_value. Для дальнейшего анализа преобразовываем в long format, чтобы каждая строка соответствовала содержала одно значение ряда и его timestamp.

In [28]:
def to_long(df, freq='h'):

    rows = []

    for _, row in df.iterrows():

        series_id = row['series_name']
        start = pd.to_datetime(row['start_timestamp'])
        values = np.asarray(row['series_value'], dtype=float)

        timestamps = pd.date_range(start=start, periods=len(values), freq=freq)

        tmp = pd.DataFrame({
            'timestamp': timestamps,
            'series_id': series_id,
            'value': values
        })
        rows.append(tmp)

    return pd.concat(rows, ignore_index=True)


df_long = (to_long(df_selected).reset_index(drop=True))
df_long_cut = (to_long(df_selected_cut).reset_index(drop=True))


In [33]:
df_long_cut.to_csv('cutted.csv', index=False)
df_long.to_csv('selected.csv', index=False)

In [13]:
df_long_sf = df_long.rename(columns={
    'series_id': 'unique_id',
    'timestamp': 'ds',
    'value': 'y',
    }
)

In [14]:
HORIZON = 168

train_df = df_long_sf.groupby('unique_id', group_keys=False).apply(lambda x: x.iloc[:-HORIZON]).reset_index(drop=True)
test_df  = df_long_sf.groupby('unique_id', group_keys=False).apply(lambda x: x.iloc[-HORIZON:]).reset_index(drop=True)

C:\Users\lenusiker\AppData\Local\Temp\ipykernel_24988\1913964870.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = df_long_sf.groupby('unique_id', group_keys=False).apply(lambda x: x.iloc[:-HORIZON]).reset_index(drop=True)
C:\Users\lenusiker\AppData\Local\Temp\ipykernel_24988\1913964870.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_df  = df_long_sf.groupby('unique_id', group_keys=False).apply(

Для почасовых рядов электропотребления естественны как минимум две сезонности: суточная с периодом 24 и недельная с периодом 168. Поэтому в бейзлайнах Seasonal Naive рассматривались оба варианта сезонной длины 24 и 168. При этом модели AutoETS и AutoTheta использовались с периодом 24, поскольку в стандартной постановке они учитывают только одну сезонность.

In [19]:
models = [
    Naive(alias='Naive'),
    SeasonalNaive(season_length=24, alias='SeasonalNaive_24'),
    SeasonalNaive(season_length=168, alias='SeasonalNaive_168'),
    AutoETS(season_length=24, alias='AutoETS_24'),
    AutoTheta(season_length=24, alias='AutoTheta_24')
]

sf = StatsForecast(models=models, freq="h", n_jobs=-1)

In [20]:
preds = sf.forecast(df=train_df, h=HORIZON)
preds.head()

,unique_id,ds,Naive,SeasonalNaive_24,SeasonalNaive_168,AutoETS_24,AutoTheta_24
0,T1,2014-12-25 00:00:01,9.0,10.0,11.0,9.457269,7.327533
1,T1,2014-12-25 01:00:01,9.0,13.0,13.0,10.414809,8.048433
2,T1,2014-12-25 02:00:01,9.0,11.0,11.0,9.584515,7.607606
3,T1,2014-12-25 03:00:01,9.0,11.0,12.0,9.852359,7.823087
4,T1,2014-12-25 04:00:01,9.0,12.0,11.0,10.102247,7.957378


In [21]:
eval_df = test_df.merge(preds, on=['unique_id', 'ds'], how='left')
eval_df.head()

,ds,unique_id,y,Naive,SeasonalNaive_24,SeasonalNaive_168,AutoETS_24,AutoTheta_24
0,2014-12-25 00:00:01,T1,11.0,9.0,10.0,11.0,9.457269,7.327533
1,2014-12-25 01:00:01,T1,9.0,9.0,13.0,13.0,10.414809,8.048433
2,2014-12-25 02:00:01,T1,10.0,9.0,11.0,11.0,9.584515,7.607606
3,2014-12-25 03:00:01,T1,10.0,9.0,11.0,12.0,9.852359,7.823087
4,2014-12-25 04:00:01,T1,11.0,9.0,12.0,11.0,10.102247,7.957378


In [22]:
def metrics(y_true, y_pred, train_values):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    train_values = np.asarray(train_values, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))

    denom = np.abs(y_true) + np.abs(y_pred)
    smape = np.mean(np.where(denom == 0, 0, 200 * np.abs(y_true - y_pred) / denom))

    scale24 = np.mean(np.abs(train_values[24:] - train_values[:-24]))
    if scale24 == 0:
        mase24 = np.nan
    else:
        mase24 = np.mean(np.abs(y_true - y_pred)) / scale24

    scale168 = np.mean(np.abs(train_values[168:] - train_values[:-168]))
    if scale168 == 0:
        mase168 = np.nan
    else:
        mase168 = np.mean(np.abs(y_true - y_pred)) / scale168

    return mae, smape, mase24, mase168

In [23]:

model_cols = ['Naive', 'SeasonalNaive_24', 'SeasonalNaive_168', 'AutoETS_24', 'AutoTheta_24']

train_map = (train_df.groupby('unique_id')['y'].apply(lambda x: np.asarray(x, dtype=float)).to_dict())

results = []

for uid, part in eval_df.groupby('unique_id'):
    part = part.sort_values('ds')

    y_true = part['y'].to_numpy()
    train_values = train_map[uid]

    for model in model_cols:
        y_pred = part[model].to_numpy()

        mae_val, smape_val, mase24_val, mase168_val = metrics(y_true=y_true, y_pred=y_pred, train_values=train_values)

        results.append({
            'series_id': uid,
            'model': model,
            'mae': mae_val,
            'smape': smape_val,
            'mase24': mase24_val,
            'mase168': mase168_val
        })

C:\Users\lenusiker\AppData\Local\Temp\ipykernel_24988\3681894441.py:9: RuntimeWarning: invalid value encountered in divide
  smape = np.mean(np.where(denom == 0, 0, 200 * np.abs(y_true - y_pred) / denom))


In [24]:
results_df = pd.DataFrame(results)
results_df.head()

,series_id,model,mae,smape,mase24,mase168
0,T1,Naive,2.916667,29.075750,0.376907,0.259885
1,T1,SeasonalNaive_24,2.720238,27.956792,0.351523,0.242382
2,T1,SeasonalNaive_168,3.482143,36.784346,0.449980,0.310270
3,T1,AutoETS_24,2.713220,27.802514,0.350616,0.241757
4,T1,AutoTheta_24,3.812199,37.576156,0.492632,0.339680


In [25]:
fin_res = (results_df.groupby('model')[['mae', 'smape', 'mase24', 'mase168']].mean().sort_values('mase24'))
fin_res

,mae,smape,mase24,mase168
model,,,,
SeasonalNaive_168,211.260615,15.828921,1.511232,1.318163
SeasonalNaive_24,315.007143,21.992882,2.120488,1.891756
AutoETS_24,378.362424,33.555222,2.406424,2.162438
AutoTheta_24,518.990109,41.068379,3.108719,2.787920
Naive,652.932837,43.039042,4.361247,3.834392
